In [1]:
from pathlib import Path
import re
import numpy as np
import xarray as xr

# =========================================================
# SETTINGS
# =========================================================
INPUT_ROOT = Path(r"Y:/Mingyue/West_Fl_Shelf/L3S_STAR/raw")
OUTPUT_ROOT = Path(r"Y:/Mingyue/West_Fl_Shelf/L3S_STAR/cropped")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

BUFFER_DEG = 0.05

points = [
    (-83.475, 25.171),
    (-83.654, 25.704),
    (-83.086, 26.010),
]

# =========================================================
# BUILD SAME SQUARE AOI IN EPSG:4326
# =========================================================
lons = [p[0] for p in points]
lats = [p[1] for p in points]

min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)

width = max_lon - min_lon
height = max_lat - min_lat
side = max(width, height) + 2 * BUFFER_DEG

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2
half_side = side / 2

square_min_lon = center_lon - half_side
square_max_lon = center_lon + half_side
square_min_lat = center_lat - half_side
square_max_lat = center_lat + half_side

print("VIIRS crop box:")
print("lon:", square_min_lon, square_max_lon)
print("lat:", square_min_lat, square_max_lat)


# =========================================================
# HELPERS
# =========================================================
def extract_first_date(name: str) -> str | None:
    m = re.search(r"(\d{8})", name)
    return m.group(1) if m else None


def get_first_existing_var(ds, candidates):
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    return None


def crop_viirs_nc(nc_path: Path, out_path: Path):
    with xr.open_dataset(nc_path, decode_timedelta=False) as ds:

        lon_var = get_first_existing_var(ds, ["lon", "longitude"])
        lat_var = get_first_existing_var(ds, ["lat", "latitude"])

        if lon_var is None or lat_var is None:
            raise RuntimeError("Cannot find lon/lat variables")

        lon = ds[lon_var].values
        lat = ds[lat_var].values

        lon = np.asarray(lon)
        lat = np.asarray(lat)

        # Convert 0–360 longitude to -180–180 if needed
        lon_fixed = np.where(lon > 180, lon - 360, lon)

        # -------------------------------------------------
        # Case 1: regular grid, lon/lat are 1D
        # -------------------------------------------------
        if lon_fixed.ndim == 1 and lat.ndim == 1:

            lon_idx = np.where(
                (lon_fixed >= square_min_lon) &
                (lon_fixed <= square_max_lon)
            )[0]

            lat_idx = np.where(
                (lat >= square_min_lat) &
                (lat <= square_max_lat)
            )[0]

            if len(lon_idx) == 0 or len(lat_idx) == 0:
                print(f"[SKIP] No overlap: {nc_path.name}")
                return

            # Get actual dimension names
            lon_dim = ds[lon_var].dims[0]
            lat_dim = ds[lat_var].dims[0]

            ds_crop = ds.isel({
                lon_dim: lon_idx,
                lat_dim: lat_idx,
            })

            # Save corrected lon values if original was 0–360
            ds_crop[lon_var] = xr.DataArray(
                lon_fixed[lon_idx],
                dims=ds[lon_var].dims,
                attrs=ds[lon_var].attrs,
            )

        # -------------------------------------------------
        # Case 2: swath/curvilinear grid, lon/lat are 2D
        # -------------------------------------------------
        elif lon_fixed.ndim == 2 and lat.ndim == 2:

            mask = (
                (lon_fixed >= square_min_lon) &
                (lon_fixed <= square_max_lon) &
                (lat >= square_min_lat) &
                (lat <= square_max_lat)
            )

            if not np.any(mask):
                print(f"[SKIP] No overlap: {nc_path.name}")
                return

            rows, cols = np.where(mask)

            r0, r1 = rows.min(), rows.max() + 1
            c0, c1 = cols.min(), cols.max() + 1

            y_dim, x_dim = ds[lon_var].dims

            ds_crop = ds.isel({
                y_dim: slice(r0, r1),
                x_dim: slice(c0, c1),
            })

            ds_crop[lon_var] = xr.DataArray(
                lon_fixed[r0:r1, c0:c1],
                dims=ds[lon_var].dims,
                attrs=ds[lon_var].attrs,
            )

        else:
            raise RuntimeError(
                f"Unsupported lon/lat shapes: lon={lon.shape}, lat={lat.shape}"
            )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        ds_crop.to_netcdf(out_path)


# =========================================================
# LOOP ALL RAW VIIRS NC FILES
# =========================================================
nc_files = sorted(INPUT_ROOT.glob("*/*/*.nc"))

print(f"Found {len(nc_files)} VIIRS files")

for nc_path in nc_files:
    try:
        date_str = extract_first_date(nc_path.name)
        if date_str is None:
            print(f"[SKIP] No date found: {nc_path.name}")
            continue

        # Preserve structure like YYYY-MM/PM/
        rel_parent = nc_path.parent.relative_to(INPUT_ROOT)
        out_dir = OUTPUT_ROOT / rel_parent
        out_path = out_dir / f"{date_str}.nc"

        if out_path.exists():
            print(f"[EXISTS] {out_path.name}")
            continue

        crop_viirs_nc(nc_path, out_path)

        print(f"[SAVED] {out_path}")

    except Exception as e:
        print(f"[ERROR] {nc_path.name}: {e}")

print("Done.")

VIIRS crop box:
lon: -83.8395 -82.90050000000001
lat: 25.121 26.06
Found 4383 VIIRS files
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200101.nc
[EXISTS] 20200101.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200102.nc
[EXISTS] 20200102.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200103.nc
[EXISTS] 20200103.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200104.nc
[EXISTS] 20200104.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200105.nc
[EXISTS] 20200105.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200106.nc
[EXISTS] 20200106.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200107.nc
[EXISTS] 20200107.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200108.nc
[EXISTS] 20200108.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200109.nc
[EXISTS] 20200109.nc
[SAVED] Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\202

In [2]:
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

# =========================================================
# SETTINGS
# =========================================================
CROPPED_ROOT = Path(r"Y:/Mingyue/West_Fl_Shelf/L3S_STAR/cropped")

TOL = 1e-10

# =========================================================
# HELPERS
# =========================================================
def get_first_existing_var(ds, candidates):
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    return None


def get_grid_info(nc_path: Path):
    with xr.open_dataset(nc_path, decode_timedelta=False) as ds:
        lon_var = get_first_existing_var(ds, ["lon", "longitude"])
        lat_var = get_first_existing_var(ds, ["lat", "latitude"])
        sst_var = get_first_existing_var(
            ds,
            ["sea_surface_temperature", "analysed_sst", "sst", "sst_subskin"]
        )

        if lon_var is None or lat_var is None or sst_var is None:
            raise RuntimeError(
                f"Missing variables: lon={lon_var}, lat={lat_var}, sst={sst_var}"
            )

        lon = np.asarray(ds[lon_var].values)
        lat = np.asarray(ds[lat_var].values)
        sst_shape = tuple(ds[sst_var].squeeze().shape)

        lon = np.where(lon > 180, lon - 360, lon)

        return {
            "file": nc_path,
            "lon_var": lon_var,
            "lat_var": lat_var,
            "sst_var": sst_var,
            "lon_shape": lon.shape,
            "lat_shape": lat.shape,
            "sst_shape": sst_shape,
            "lon": lon,
            "lat": lat,
        }


# =========================================================
# LOAD ALL CROPPED FILES
# =========================================================
nc_files = sorted(CROPPED_ROOT.glob("*/*/*.nc"))

print(f"Found cropped VIIRS files: {len(nc_files)}")

if len(nc_files) == 0:
    raise RuntimeError(f"No cropped .nc files found under: {CROPPED_ROOT}")

# Use first file as reference
ref = get_grid_info(nc_files[0])

print("\nReference file:")
print(ref["file"])
print("lon_shape:", ref["lon_shape"])
print("lat_shape:", ref["lat_shape"])
print("sst_shape:", ref["sst_shape"])
print("lon range:", np.nanmin(ref["lon"]), np.nanmax(ref["lon"]))
print("lat range:", np.nanmin(ref["lat"]), np.nanmax(ref["lat"]))

rows = []
bad_files = []

for i, nc_path in enumerate(nc_files, start=1):
    try:
        info = get_grid_info(nc_path)

        same_lon_shape = info["lon_shape"] == ref["lon_shape"]
        same_lat_shape = info["lat_shape"] == ref["lat_shape"]
        same_sst_shape = info["sst_shape"] == ref["sst_shape"]

        same_lon_grid = (
            same_lon_shape and
            np.allclose(info["lon"], ref["lon"], equal_nan=True, atol=TOL)
        )

        same_lat_grid = (
            same_lat_shape and
            np.allclose(info["lat"], ref["lat"], equal_nan=True, atol=TOL)
        )

        same_all = (
            same_lon_shape
            and same_lat_shape
            and same_sst_shape
            and same_lon_grid
            and same_lat_grid
        )

        rows.append({
            "file": str(nc_path),
            "same_all": same_all,
            "same_lon_shape": same_lon_shape,
            "same_lat_shape": same_lat_shape,
            "same_sst_shape": same_sst_shape,
            "same_lon_grid": same_lon_grid,
            "same_lat_grid": same_lat_grid,
            "lon_shape": info["lon_shape"],
            "lat_shape": info["lat_shape"],
            "sst_shape": info["sst_shape"],
        })

        if not same_all:
            bad_files.append(nc_path)
            print(f"[MISMATCH] {nc_path.name}")

        if i % 100 == 0:
            print(f"Checked {i}/{len(nc_files)}")

    except Exception as e:
        bad_files.append(nc_path)
        rows.append({
            "file": str(nc_path),
            "same_all": False,
            "error": str(e),
        })
        print(f"[ERROR] {nc_path.name}: {e}")

df_check = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("GRID CHECK SUMMARY")
print("=" * 70)
print("Total files:", len(nc_files))
print("Files matching reference:", int(df_check["same_all"].sum()))
print("Files mismatching/error:", len(bad_files))

if len(bad_files) == 0:
    print("\n✅ All cropped VIIRS files have the same shape and lon/lat grid.")
else:
    print("\n⚠️ Some files do NOT match the reference grid.")
    display(df_check[df_check["same_all"] == False].head(30))

# Optional save report
REPORT_PATH = CROPPED_ROOT / "cropped_viirs_grid_check_report.csv"
df_check.to_csv(REPORT_PATH, index=False)
print(f"\nSaved report to: {REPORT_PATH}")

Found cropped VIIRS files: 2192

Reference file:
Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\2020-01\PM\20200101.nc
lon_shape: (47,)
lat_shape: (47,)
sst_shape: (47, 47)
lon range: -83.83 -82.91
lat range: 25.13 26.05
Checked 100/2192
Checked 200/2192
Checked 300/2192
Checked 400/2192
Checked 500/2192
Checked 600/2192
Checked 700/2192
Checked 800/2192
Checked 900/2192
Checked 1000/2192
Checked 1100/2192
Checked 1200/2192
Checked 1300/2192
Checked 1400/2192
Checked 1500/2192
Checked 1600/2192
Checked 1700/2192
Checked 1800/2192
Checked 1900/2192
Checked 2000/2192
Checked 2100/2192

GRID CHECK SUMMARY
Total files: 2192
Files matching reference: 2192
Files mismatching/error: 0

✅ All cropped VIIRS files have the same shape and lon/lat grid.

Saved report to: Y:\Mingyue\West_Fl_Shelf\L3S_STAR\cropped\cropped_viirs_grid_check_report.csv


## crop if whole landsat scene included

lat and lon are getting from crop_viirs_landsat_whole_scene.ipynb - x20 buffer if exist 

In [2]:
from pathlib import Path
import re
import numpy as np
import xarray as xr

# =========================================================
# SETTINGS
# =========================================================
INPUT_ROOT = Path(r"../Timor_part1/L3S_STAR/raw")
OUTPUT_ROOT = Path(r"../Timor_part1/L3S_STAR/cropped")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Crop ONLY daytime files
KEEP_TAG = "_D-ACSPO_"

# =========================================================
# WHOLE LANDSAT SCENE + 20px BUFFER BOUNDS
# EPSG:4326: west, south, east, north
# =========================================================
square_min_lon = 124.13215391013588
square_min_lat = -9.743977465140459
square_max_lon = 126.23502593648494
square_max_lat = -7.614350046652487

print("VIIRS crop box:")
print("lon:", square_min_lon, square_max_lon)
print("lat:", square_min_lat, square_max_lat)
print("Keeping only files with:", KEEP_TAG)


# =========================================================
# HELPERS
# =========================================================
def extract_first_date(name: str) -> str | None:
    m = re.search(r"(\d{8})", name)
    return m.group(1) if m else None


def get_first_existing_var(ds, candidates):
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    return None


def crop_viirs_nc(nc_path: Path, out_path: Path):
    with xr.open_dataset(nc_path, decode_timedelta=False) as ds:

        lon_var = get_first_existing_var(ds, ["lon", "longitude"])
        lat_var = get_first_existing_var(ds, ["lat", "latitude"])

        if lon_var is None or lat_var is None:
            raise RuntimeError("Cannot find lon/lat variables")

        lon = np.asarray(ds[lon_var].values)
        lat = np.asarray(ds[lat_var].values)

        lon_fixed = np.where(lon > 180, lon - 360, lon)

        # -------------------------------------------------
        # Case 1: regular grid, lon/lat are 1D
        # -------------------------------------------------
        if lon_fixed.ndim == 1 and lat.ndim == 1:

            lon_idx = np.where(
                (lon_fixed >= square_min_lon) &
                (lon_fixed <= square_max_lon)
            )[0]

            lat_idx = np.where(
                (lat >= square_min_lat) &
                (lat <= square_max_lat)
            )[0]

            if len(lon_idx) == 0 or len(lat_idx) == 0:
                print(f"[SKIP] No overlap: {nc_path.name}")
                return False

            lon_dim = ds[lon_var].dims[0]
            lat_dim = ds[lat_var].dims[0]

            ds_crop = ds.isel({
                lon_dim: lon_idx,
                lat_dim: lat_idx,
            })

            ds_crop[lon_var] = xr.DataArray(
                lon_fixed[lon_idx],
                dims=ds[lon_var].dims,
                attrs=ds[lon_var].attrs,
            )

        # -------------------------------------------------
        # Case 2: swath/curvilinear grid, lon/lat are 2D
        # -------------------------------------------------
        elif lon_fixed.ndim == 2 and lat.ndim == 2:

            mask = (
                (lon_fixed >= square_min_lon) &
                (lon_fixed <= square_max_lon) &
                (lat >= square_min_lat) &
                (lat <= square_max_lat)
            )

            if not np.any(mask):
                print(f"[SKIP] No overlap: {nc_path.name}")
                return False

            rows, cols = np.where(mask)

            r0, r1 = rows.min(), rows.max() + 1
            c0, c1 = cols.min(), cols.max() + 1

            y_dim, x_dim = ds[lon_var].dims

            ds_crop = ds.isel({
                y_dim: slice(r0, r1),
                x_dim: slice(c0, c1),
            })

            ds_crop[lon_var] = xr.DataArray(
                lon_fixed[r0:r1, c0:c1],
                dims=ds[lon_var].dims,
                attrs=ds[lon_var].attrs,
            )

        else:
            raise RuntimeError(
                f"Unsupported lon/lat shapes: lon={lon.shape}, lat={lat.shape}"
            )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        ds_crop.to_netcdf(out_path)

        return True


# =========================================================
# LOOP DAYTIME RAW VIIRS NC FILES ONLY
# =========================================================
all_nc_files = sorted(INPUT_ROOT.glob("*/*/*.nc"))

nc_files = [
    p for p in all_nc_files
    if KEEP_TAG in p.name
]

print(f"Found total VIIRS files: {len(all_nc_files)}")
print(f"Found daytime files ({KEEP_TAG}): {len(nc_files)}")

saved = 0
skipped = 0
failed = []

for nc_path in nc_files:
    try:
        date_str = extract_first_date(nc_path.name)

        if date_str is None:
            print(f"[SKIP] No date found: {nc_path.name}")
            skipped += 1
            continue

        # Preserve folder structure like YYYY-MM/AM or YYYY-MM/PM
        rel_parent = nc_path.parent.relative_to(INPUT_ROOT)
        out_dir = OUTPUT_ROOT / rel_parent
        out_path = out_dir / f"{date_str}_D.nc"

        if out_path.exists():
            print(f"[EXISTS] {out_path}")
            skipped += 1
            continue

        ok = crop_viirs_nc(nc_path, out_path)

        if ok:
            print(f"[SAVED] {out_path}")
            saved += 1
        else:
            skipped += 1

    except Exception as e:
        print(f"[ERROR] {nc_path.name}: {e}")
        failed.append((nc_path.name, str(e)))

print("=" * 80)
print("Done.")
print("Saved:", saved)
print("Skipped:", skipped)
print("Failed:", len(failed))

if failed:
    print("\nFailed files:")
    for name, err in failed:
        print(name, "->", err)

VIIRS crop box:
lon: 124.13215391013588 126.23502593648494
lat: -9.743977465140459 -7.614350046652487
Keeping only files with: _D-ACSPO_
Found total VIIRS files: 4383
Found daytime files (_D-ACSPO_): 2191
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200101_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200102_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200103_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200104_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200105_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200106_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200107_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200108_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200109_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200110_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200111_D.nc
[SAVED] ../Timor_part1/L3S_STAR/cropped/2020-01/PM/20200112_D.nc
[SAVED] ../Timo